# AquaPINN Sequim Quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cwq2016/AquaPINN_tmp/blob/main/Examples/Colab/01_Sequim_Quickstart.ipynb)

Run the Sequim example in four steps: set up the notebook, choose the inputs and tracking settings, run the pipeline, and view the track. Results are saved in a new run folder.


## 1. Set up

Run this cell once. Colab saves results to Google Drive; local Jupyter saves them in `AquaPINN_Colab_Runs`.


In [ ]:
%pip -q install "numpy<2.5" scipy pandas matplotlib utm networkx scikit-learn tqdm six tomli-w

from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/cwq2016/AquaPINN_tmp.git"
REPO_REF = "main"
REPO_DIR = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "AquaPINN_Tracker3D").is_dir()), None)
if REPO_DIR is None:
    REPO_DIR = Path("/content/AquaPINN_tmp")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF,
                        REPO_URL, str(REPO_DIR)], check=True)
sys.path.insert(0, str(REPO_DIR / "Examples/Colab"))
from colab_support import prepare_output_folder, run_sequim
OUTPUT_PARENT = prepare_output_folder(REPO_DIR)
print("Ready. Results will be saved under:", OUTPUT_PARENT)


## 2. Choose inputs and tracking settings

Change the paths if you want to use files in Google Drive. TagID 2 is the supplied Sequim desktop example tag. The available methods are AML, NN, and NNplus (repeated NN fits).


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Input files: replace these paths with Drive paths for your own data.
MAT_FILE = REPO_DIR / "Examples/Sequim/data/DriftingTags_timeCorrected.mat"
PHONE_FILE = REPO_DIR / "Examples/Sequim/config/sequim_testing_node_locs_w_beacons.csv"
TAG_FILE = REPO_DIR / "Examples/Sequim/config/DriftingTags.csv"
TEST_FILE = REPO_DIR / "Examples/Sequim/config/EA_test_description.csv"
GPS_FILE = REPO_DIR / "Examples/Sequim/config/drifting_GPS_day1_local_system.csv"

# Main tracking settings
METHOD = "AML"  # AML, NN, or NNplus
NNPLUS_SAMPLES = 5
TIME_START = "2024-2-21 11:30:00"
TIME_END = "2024-2-21 13:30:00"
TIME_ZONE_HOURS = 0
TEMPERATURE_C = 8.0
HORIZONTAL_ONLY = True
DEVICE = "Automatic"  # Automatic or CPU only

display(pd.DataFrame({"Setting": ["Tag ID", "Method", "Time range", "Timezone", "Temperature (C)", "2D only", "Device"],
                        "Value": [2, METHOD, f"{TIME_START} to {TIME_END}", TIME_ZONE_HOURS, TEMPERATURE_C, HORIZONTAL_ONLY, DEVICE]}))


## 3. Run Sequim

This uses the existing Sequim conversion and tracking workflow. The corrected detections do not need a new synchronization run.


In [ ]:
RUN_DIR, TRACK_CSV = run_sequim(
    repo_dir=REPO_DIR, output_parent=OUTPUT_PARENT,
    mat_file=MAT_FILE, phone_file=PHONE_FILE, tag_file=TAG_FILE,
    test_file=TEST_FILE, gps_file=GPS_FILE,
    method=METHOD, nnplus_samples=NNPLUS_SAMPLES,
    time_start=TIME_START, time_end=TIME_END, time_zone_hours=TIME_ZONE_HOURS,
    temperature_c=TEMPERATURE_C, horizontal_only=HORIZONTAL_ONLY, device=DEVICE,
)
print("Run complete. Results:", RUN_DIR)


## 4. View results

The run folder contains the track CSV and pickle sidecar, plots, parameters, log, and a desktop Simple Example run record. Keep the whole folder if you want to reopen the run later.


In [ ]:
import pandas as pd
from IPython.display import display, Image

track = pd.read_csv(TRACK_CSV)
display(track.head())

for figure in sorted((RUN_DIR / "results").rglob("*.jpg"))[:4]:
    display(Image(filename=str(figure)))

print("Saved run folder:", RUN_DIR)
print("Desktop run record:", RUN_DIR / "run.json")
